- 1. Use lat/lon columns for spatial clustering (KMeans or DBSCAN).
- 2. Optionally aggregate by cleaned_address or city/state for region assignment.
- 3. Assign sales people to regions with highest cluster density.

- Assuming that ship_to_address is not unique, more than one customer "live" at the same location.

In [0]:
unique_count = df.select("ship_to_address").distinct().count()
total_count = df.count()
print("All ship_to_address are unique:", unique_count == total_count)

In [0]:
from pyspark.sql import functions as F

df = spark.sql("SELECT * FROM end_to_end_retail.db_gold.tbg_customer_address_clustering")

dup_df = df.groupBy("ship_to_address").agg(
    F.countDistinct("lat", "lon").alias("unique_lat_lon_count")
).filter("unique_lat_lon_count > 1")

display(df.join(dup_df, "ship_to_address"))

In [0]:
df_customer_address_clustering = spark.sql("""
  SELECT
    customer_id,
    ship_to_address,
    state,
    city,
    split(ship_to_address, ',')[1] AS zipcode,
    lon,
    lat,
    concat_ws(',', slice(split(ship_to_address, ','), 3, size(split(ship_to_address, ','))-2)) AS ship_to_address_remainder
  FROM end_to_end_retail.db_gold.tbg_customer_address_clustering
  WHERE NOT (split(ship_to_address, ',')[0] LIKE "%NO SITUS%" OR split(ship_to_address, ',')[0] LIKE "%UNASSIGNED%")
""")

- Since DBSCAN relies on a single distance matrix, we will:
- Calculate a Spatial Distance Matrix (Haversine).
- Calculate a Textual Similarity Matrix (TF-IDF + Cosine Distance).
- Hybridize them: Only cluster points that are both physically close and textually similar.
- Refine: Hard-partition the results by Zip Code.

In [0]:
%sql
  SELECT
    COUNT(*)
  FROM end_to_end_retail.db_gold.tbg_customer_address_clustering
  WHERE NOT (split(ship_to_address, ',')[0] LIKE "%NO SITUS%" OR split(ship_to_address, ',')[0] LIKE "%UNASSIGNED%")

In [0]:
%pip install kneed scikit-learn pandas

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances, haversine_distances
from sklearn.cluster import DBSCAN
from kneed import KneeLocator

# 1. Load Data (Assuming 'df' is your Spark DataFrame)
# We need: street_address, zipcode, latitude, longitude
raw_data = df.select("street_address", "zipcode", "latitude", "longitude").toPandas()

# 2. Text Vectorization (TF-IDF)
# We use n-grams (3-char chunks) to handle typos in street names
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 3))
tfidf_matrix = tfidf.fit_transform(raw_data['street_address'].fillna(''))

# 3. Calculate Hybrid Distance Matrix
# Convert Lat/Lon to Radians
coords_radians = np.radians(raw_data[['latitude', 'longitude']].values)

# Calculate physical distance in Radians
dist_spatial = haversine_distances(coords_radians)

# Calculate textual distance (1 - similarity)
dist_textual = cosine_distances(tfidf_matrix)

# Hybrid Distance: Adjust weights as needed (e.g., 70% Spatial, 30% Textual)
# This ensures that even if addresses are nearby, if the names are totally different, 
# the distance increases.
dist_hybrid = (0.7 * dist_spatial) + (0.3 * dist_textual)

# 4. Automated Epsilon Detection on the Hybrid Matrix
min_samples = 5
neighbors = np.sort(dist_hybrid, axis=1)
k_distances = neighbors[:, min_samples]

kneedle = KneeLocator(
    range(len(k_distances)), 
    np.sort(k_distances), 
    curve='convex', 
    direction='increasing'
)
optimal_eps = kneedle.knee_y or np.percentile(k_distances, 90)

# 5. Run DBSCAN on Hybrid Matrix
db = DBSCAN(
    eps=optimal_eps, 
    min_samples=min_samples, 
    metric='precomputed' # Critical: we calculated our own matrix
).fit(dist_hybrid)

raw_data['initial_cluster'] = db.labels_

# 6. Refinement: Hard Partition by Zip Code
# This ensures a cluster cannot span across different Zip Codes 
# even if the spatial/textual distance was small.
raw_data['refined_cluster'] = (
    raw_data['initial_cluster'].astype(str) + "_" + raw_data['zipcode'].astype(str)
)

# Re-map noise (-1) back to -1
raw_data.loc[raw_data['initial_cluster'] == -1, 'refined_cluster'] = -1

# 7. Convert back to Spark
final_df = spark.createDataFrame(raw_data)
display(final_df)


In [0]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from kneed import KneeLocator # For automated elbow detection

# 1. Prepare Data
# Assuming 'df' is your Spark DataFrame
pd_df = df.select("lat", "lon").toPandas()
coords_radians = np.radians(pd_df.to_numpy())

# 2. Determine Optimal Epsilon (Distance Threshold)
# min_samples is usually (Dimensions * 2) or chosen by domain knowledge
min_samples = 5 

# Calculate the distance to the n-th nearest neighbor
neighbors = NearestNeighbors(n_neighbors=min_samples, metric='haversine')
neighbors_fit = neighbors.fit(coords_radians)
distances, indices = neighbors_fit.kneighbors(coords_radians)

# Sort distances (the k-th distance) to find the elbow
sort_distances = np.sort(distances[:, min_samples-1], axis=0)

# 3. Automated Elbow Detection
# S=1.0 is sensitivity; curve is 'convex' because distances increase
kneedle = KneeLocator(
    range(len(sort_distances)), 
    sort_distances, 
    curve='convex', 
    direction='increasing'
)

optimal_eps = kneedle.knee_y

if optimal_eps is None or optimal_eps == 0:
    print("Could not find elbow. Defaulting to a small physical distance.")
    optimal_eps = 0.5 / 6371.0088 # Default 500m
else:
    print(f"Optimal Epsilon found: {optimal_eps:.6f} radians " 
          f"({optimal_eps * 6371.0088:.4f} km)")

# 4. Run DBSCAN with Optimized Parameters
db = DBSCAN(
    eps=optimal_eps, 
    min_samples=min_samples, 
    metric='haversine', 
    algorithm='ball_tree'
).fit(coords_radians)

pd_df['cluster_id'] = db.labels_

# 5. Bring results back to Spark
final_df = spark.createDataFrame(pd_df)
display(final_df)


In [0]:
# Apply TfidfVectorizer to ship_to_address
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
tokenizer = Tokenizer(inputCol="ship_to_address", outputCol="words")
wordsData = tokenizer.transform(df)
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20)
featurizedData = hashingTF.transform(wordsData)
idf = IDF(inputCol="rawFeatures", outputCol="tfidfFeatures")
idfModel = idf.fit(featurizedData)
tfidfData = idfModel.transform(featurizedData)

# Combine tfidfFeatures with lat/lon
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols=["tfidfFeatures", "lat", "lon"],
    outputCol="features"
)
final_df = assembler.transform(tfidfData)

df_customer_address_clustering

In [0]:
# Apply DBSCAN clustering using sklearn
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt

features_pd = final_df.select("ship_to_address", "lat", "lon", "features").toPandas()
X = features_pd["features"].apply(lambda v: v.toArray()).tolist()
dbscan = DBSCAN(eps=0.5, min_samples=5)
features_pd["cluster"] = dbscan.fit_predict(X)

# Remove outliers (DBSCAN assigns -1 to outliers)
features_pd_no_outliers = features_pd[features_pd["cluster"] != -1]

# Calculate silhouette score for non-outlier clusters
if features_pd_no_outliers["cluster"].nunique() > 1:
    sil_score = silhouette_score(
        features_pd_no_outliers["features"].apply(lambda v: v.toArray()).tolist(),
        features_pd_no_outliers["cluster"]
    )
else:
    sil_score = None

# Show clustered results without outliers
display(spark.createDataFrame(features_pd_no_outliers[["ship_to_address", "lat", "lon", "cluster"]]))

# Visualize clusters without outliers and highlight main clusters
plt.figure(figsize=(8,6))
scatter = plt.scatter(
    features_pd_no_outliers["lat"],
    features_pd_no_outliers["lon"],
    c=features_pd_no_outliers["cluster"],
    cmap="tab10",
    edgecolor='k'
)
plt.xlabel("Latitude")
plt.ylabel("Longitude")
plt.title(f"DBSCAN Clusters (Outliers Removed)\nSilhouette Score: {sil_score:.2f}" if sil_score else "DBSCAN Clusters (Outliers Removed)")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_gold.tbg_customer_address_clustering limit 5